CONFIGURATION

In [4]:
!pip install -q google-adk litellm google-cloud-aiplatform google-cloud-secret-manager google-cloud-api-keys requests

In [3]:
import json
import os
import textwrap
from typing import Any, Dict, List, Optional

import requests

In [5]:
# --- Google Cloud / Vertex AI -------------------------------------------------
import getpass
import subprocess

import google.auth


def _detect_project_id() -> Optional[str]:
    """
    Determine the active GCP project without hardcoding it.

    Checks, in order: an explicit override in the environment, the project
    embedded in Application Default Credentials (already set in Colab,
    Colab Enterprise, Vertex AI Workbench, and Cloud Shell, or via
    `gcloud auth application-default login` on a local machine), and
    finally the active `gcloud` CLI config.

    Returns:
        Optional[str]: The detected project ID, or None if none was found.
    """
    for var in ("GOOGLE_CLOUD_PROJECT", "GCLOUD_PROJECT", "GCP_PROJECT"):
        if os.environ.get(var):
            return os.environ[var]
    try:
        _, project_id = google.auth.default()
        if project_id:
            return project_id
    except Exception:
        pass
    try:
        result = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True,
            text=True,
            timeout=10,
            check=True,
        )
        value = result.stdout.strip()
        if value and value != "(unset)":
            return value
    except Exception:
        pass
    return None


PROJECT_ID = _detect_project_id() or "your-gcp-project-id"  # <-- set manually if auto-detection fails
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

if PROJECT_ID == "your-gcp-project-id":
    print(
        "WARNING: could not auto-detect a GCP project. Set PROJECT_ID manually "
        "above, or run `gcloud auth application-default login` / "
        "`gcloud config set project <id>`."
    )

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"   # route Gemini calls through Vertex AI

# --- Models ------------------------------------------------------------------
MODEL_GEMINI_FLASH = "gemini-2.5-flash"

# Anthropic Claude via Vertex AI Model Garden -- billed to and authenticated
# with this same GCP project, so no separate Anthropic API key is needed.
# Confirmed from the model card's sample code (AnthropicVertex client) for
# this project: model ID "claude-sonnet-5", served from the "global" endpoint.
MODEL_CLAUDE = "vertex_ai/claude-sonnet-5"
CLAUDE_LOCATION = os.environ.get("CLAUDE_LOCATION", "global")

# --- API keys ----------------------------------------------------------------
def _ensure_apis_enabled(*service_names: str) -> None:
    """
    Best-effort, idempotent enabling of GCP APIs this notebook depends on.

    A fresh project (a new qwiklabs sandbox, for instance) has Secret
    Manager and the API Keys API turned off by default, which otherwise
    surfaces as a `PermissionDenied: SERVICE_DISABLED` error the first time
    anyone runs this notebook. Enabling an already-enabled API is a fast
    no-op, so calling this unconditionally on every run is safe. Any
    failure (missing IAM, no `gcloud` binary) is swallowed -- the rest of
    the secret-loading flow still works if the APIs happened to be on
    already, and falls back to a prompt otherwise.

    Args:
        *service_names (str): Fully qualified service names, e.g.
            "secretmanager.googleapis.com".
    """
    try:
        subprocess.run(
            ["gcloud", "services", "enable", *service_names, "--project", PROJECT_ID],
            capture_output=True,
            timeout=60,
        )
    except Exception:
        pass


_ensure_apis_enabled("secretmanager.googleapis.com", "apikeys.googleapis.com")


def _discover_maps_api_key() -> Optional[str]:
    """
    Find an API key already created in this project via the GCP Console.

    Uses the API Keys API (`apikeys.googleapis.com`) to list keys in the
    active project and fetch the key string of one restricted to a Maps
    Platform service (Geocoding, Maps, etc.), or an unrestricted key if no
    Maps-restricted one is found. This is what lets the notebook pick up a
    key you created through *APIs & Services → Credentials* without typing
    it in anywhere. Prints the reason for a miss at each step so a failure
    here is self-diagnosing rather than a silent fall-through to the prompt.

    Returns:
        Optional[str]: The discovered key string, or None if no usable key
            was found or the API Keys API is not reachable.
    """
    try:
        from google.cloud import api_keys_v2
    except ImportError:
        print("  (google-cloud-api-keys not installed -- run cell-02, then restart the runtime.)")
        return None
    try:
        client = api_keys_v2.ApiKeysClient()
        keys = list(client.list_keys(parent=f"projects/{PROJECT_ID}/locations/global"))
    except Exception as exc:
        print(f"  (Could not list API keys in {PROJECT_ID}: {type(exc).__name__}: {exc})")
        return None
    if not keys:
        print(
            f"  (No API keys found in {PROJECT_ID} -- create one under "
            "APIs & Services -> Credentials.)"
        )
        return None

    def _is_maps_related(key) -> bool:
        targets = getattr(getattr(key, "restrictions", None), "api_targets", [])
        if not targets:
            return True  # Unrestricted key -- usable for Geocoding too.
        return any(
            "geocoding" in target.service.lower() or "maps" in target.service.lower()
            for target in targets
        )

    ordered = sorted(keys, key=_is_maps_related, reverse=True)
    last_error: Optional[Exception] = None
    for key in ordered:
        try:
            response = client.get_key_string(name=key.name)
            if response.key_string:
                return response.key_string
        except Exception as exc:
            last_error = exc
            continue
    if last_error is not None:
        print(
            f"  (Found {len(ordered)} key(s) in {PROJECT_ID} but could not read a key "
            f"string: {type(last_error).__name__}: {last_error})"
        )
    return None


def _persist_to_secret_manager(name: str, value: str) -> bool:
    """
    Best-effort save of a secret value to Secret Manager for future runs.

    Creates the secret if it does not exist and adds `value` as its latest
    version. Any failure (insufficient IAM, disabled billing, no network)
    is swallowed: the caller already has `value` for the current session
    regardless of whether persistence succeeds.

    Args:
        name (str): Secret ID to create or add a version to.
        value (str): Secret value to store.

    Returns:
        bool: True if the value was persisted, False otherwise.
    """
    try:
        from google.api_core.exceptions import AlreadyExists
        from google.cloud import secretmanager
    except ImportError:
        return False
    try:
        client = secretmanager.SecretManagerServiceClient()
        parent = f"projects/{PROJECT_ID}"
        try:
            client.create_secret(
                request={
                    "parent": parent,
                    "secret_id": name,
                    "secret": {"replication": {"automatic": {}}},
                }
            )
        except AlreadyExists:
            pass
        client.add_secret_version(
            request={
                "parent": f"{parent}/secrets/{name}",
                "payload": {"data": value.encode("utf-8")},
            }
        )
        return True
    except Exception:
        return False


def _load_secret(name: str, prompt_if_missing: bool = True) -> Optional[str]:
    """
    Load a secret, discovering or prompting for it if none is found yet.

    Tries, in order: `google.colab.userdata` (classic Colab's secrets
    panel), a GCP Secret Manager secret named `name` in the active project
    (works anywhere with Secret Manager access via Application Default
    Credentials — Colab Enterprise, Vertex AI Workbench, Cloud Shell, or a
    local `gcloud auth application-default login`), a plain environment
    variable, and — for `GOOGLE_MAPS_API_KEY` specifically — an existing
    API key already created in the project via the GCP Console. If none of
    those have it and `prompt_if_missing` is True, prompts for the value
    through a masked input — so it never appears in notebook source,
    notebook output, or anywhere outside this kernel — and saves it to
    Secret Manager so later runs, by anyone with access to this project,
    never have to enter it again.

    Args:
        name (str): Name of the secret / environment variable to load.
        prompt_if_missing (bool): Whether to interactively prompt for the
            value when it cannot be found anywhere else. Set False for
            optional secrets that a run should silently skip rather than
            block on.

    Returns:
        Optional[str]: The secret value, or None if it could not be obtained.
    """
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    try:
        from google.cloud import secretmanager

        client = secretmanager.SecretManagerServiceClient()
        secret_path = f"projects/{PROJECT_ID}/secrets/{name}/versions/latest"
        response = client.access_secret_version(name=secret_path)
        value = response.payload.data.decode("utf-8").strip()
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        return value
    if name == "GOOGLE_MAPS_API_KEY":
        value = _discover_maps_api_key()
        if value:
            _persist_to_secret_manager(name, value)  # Cache so later runs skip discovery too.
            return value
    if not prompt_if_missing:
        return None
    value = getpass.getpass(f"{name} not found — paste it now (input hidden): ").strip()
    if not value:
        return None
    if _persist_to_secret_manager(name, value):
        print(f"Saved {name} to Secret Manager — future runs won't prompt for it again.")
    else:
        print(f"Could not save {name} to Secret Manager; using it for this session only.")
    return value


GOOGLE_MAPS_API_KEY = _load_secret("GOOGLE_MAPS_API_KEY") or ""

if GOOGLE_MAPS_API_KEY:
    os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY

print(f"Project:          {PROJECT_ID}")
print(f"Location:         {LOCATION}")
print(f"Maps key loaded:  {bool(GOOGLE_MAPS_API_KEY)}")

Project:          qwiklabs-gcp-04-c3dfd38bd121
Location:         us-central1
Maps key loaded:  True


TOOLS

In [6]:
NWS_API_BASE = "https://api.weather.gov"
GEOCODE_API_URL = "https://maps.googleapis.com/maps/api/geocode/json"

# The NWS API requires a descriptive User-Agent identifying the caller.
USER_AGENT = "adk-skills-workshop-weather-agent (contact: your.email@example.com)"
NWS_HEADERS = {"User-Agent": USER_AGENT, "Accept": "application/geo+json"}
REQUEST_TIMEOUT = 20

National Weather Service Functions

In [7]:
def get_weather_forecast(lat: float, lon: float) -> Dict[str, Any]:
    """
    Fetch current conditions and the extended forecast for a US location.

    Queries the U.S. National Weather Service (NWS) API, which requires a
    two-step lookup: the /points endpoint maps coordinates to a forecast grid,
    and the forecast URL it returns provides the forecast periods themselves.
    Coverage is limited to the United States and its territories.

    Args:
        lat (float): Latitude in decimal degrees (e.g., 38.8977).
        lon (float): Longitude in decimal degrees (e.g., -77.0365).

    Returns:
        Dict[str, Any]: On success, a dictionary with keys:
            "status" (str): The literal "success".
            "location" (str): Nearest city and state per the NWS.
            "current" (Dict[str, str]): The current forecast period, with keys
                "name", "temperature", "temperature_unit", "wind",
                "short_forecast", and "detailed_forecast".
            "forecast" (List[Dict[str, str]]): Up to eight upcoming periods in
                the same shape as "current".
        On failure, a dictionary with keys:
            "status" (str): The literal "error".
            "error_message" (str): Human-readable explanation of the failure.
    """
    def _summarize(period: Dict[str, Any]) -> Dict[str, str]:
        """Flatten one NWS forecast period into plain strings for the model."""
        wind = f"{period.get('windSpeed', '')} {period.get('windDirection', '')}"
        return {
            "name": period.get("name", ""),
            "temperature": str(period.get("temperature", "")),
            "temperature_unit": period.get("temperatureUnit", ""),
            "wind": wind.strip(),
            "short_forecast": period.get("shortForecast", ""),
            "detailed_forecast": period.get("detailedForecast", ""),
        }

    try:
        points_response = requests.get(
            f"{NWS_API_BASE}/points/{lat:.4f},{lon:.4f}",
            headers=NWS_HEADERS,
            timeout=REQUEST_TIMEOUT,
        )
        points_response.raise_for_status()
        properties = points_response.json()["properties"]

        forecast_response = requests.get(
            properties["forecast"], headers=NWS_HEADERS, timeout=REQUEST_TIMEOUT
        )
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
    except requests.RequestException as exc:
        return {
            "status": "error",
            "error_message": (
                f"NWS forecast request failed for ({lat}, {lon}): {exc}. "
                "The NWS API only covers the United States and its territories."
            ),
        }
    except (KeyError, ValueError) as exc:
        return {
            "status": "error",
            "error_message": f"Unexpected NWS response format: {exc}",
        }

    if not periods:
        return {
            "status": "error",
            "error_message": f"The NWS returned no forecast periods for ({lat}, {lon}).",
        }

    relative = properties.get("relativeLocation", {}).get("properties", {})
    city, state = relative.get("city"), relative.get("state")
    area = f"{city}, {state}" if city and state else f"{lat}, {lon}"

    return {
        "status": "success",
        "location": area,
        "current": _summarize(periods[0]),
        "forecast": [_summarize(period) for period in periods[1:9]],
    }

In [8]:
def get_active_weather_alerts(state_code: str) -> Dict[str, Any]:
    """
    Retrieve active National Weather Service alerts for a US state.

    Returns watches, warnings, and advisories currently in effect, which the
    agent uses to escalate a routine forecast into a weather alert.

    Args:
        state_code (str): Two-letter US state or territory code (e.g., "TX", "FL").

    Returns:
        Dict[str, Any]: On success, a dictionary with keys:
            "status" (str): The literal "success".
            "state" (str): The uppercase state code that was queried.
            "alert_count" (int): Total number of active alerts found.
            "alerts" (List[Dict[str, str]]): Up to ten alerts, each with keys
                "event", "severity", "urgency", "areas", "headline", and
                "instruction".
        On failure, a dictionary with keys:
            "status" (str): The literal "error".
            "error_message" (str): Human-readable explanation of the failure.
    """
    code = state_code.strip().upper()
    if len(code) != 2 or not code.isalpha():
        return {
            "status": "error",
            "error_message": (
                f"'{state_code}' is not a two-letter US state code. "
                "Use a value such as 'CO' or 'FL'."
            ),
        }

    try:
        response = requests.get(
            f"{NWS_API_BASE}/alerts/active",
            params={"area": code},
            headers=NWS_HEADERS,
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        features = response.json().get("features", [])
    except requests.RequestException as exc:
        return {
            "status": "error",
            "error_message": f"NWS alerts request failed for {code}: {exc}",
        }
    except ValueError as exc:
        return {
            "status": "error",
            "error_message": f"Unexpected NWS alerts response format: {exc}",
        }

    alerts: List[Dict[str, str]] = []
    for feature in features[:10]:
        props = feature.get("properties", {})
        alerts.append(
            {
                "event": props.get("event", ""),
                "severity": props.get("severity", ""),
                "urgency": props.get("urgency", ""),
                "areas": props.get("areaDesc", ""),
                "headline": props.get("headline", ""),
                "instruction": (props.get("instruction") or "")[:400],
            }
        )

    return {
        "status": "success",
        "state": code,
        "alert_count": len(features),
        "alerts": alerts,
    }

Google Services

In [9]:
def get_lat_lon(location: str) -> Dict[str, Any]:
    """
    Convert a human-readable place name into geographic coordinates.

    Uses the Google Maps Geocoding API to resolve a free-form location string
    (for example, "Austin, TX" or "1600 Pennsylvania Ave NW, Washington DC")
    into latitude and longitude, which the weather tools require.

    Args:
        location (str): Free-form place name, address, or "City, State" string.

    Returns:
        Dict[str, Any]: On success, a dictionary with keys:
            "status" (str): The literal "success".
            "location" (str): Normalized formatted address from Google.
            "lat" (float): Latitude in decimal degrees.
            "lon" (float): Longitude in decimal degrees.
            "state_code" (str): Two-letter US state code, or "" if not found.
        On failure, a dictionary with keys:
            "status" (str): The literal "error".
            "error_message" (str): Human-readable explanation of the failure.
    """
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    if not api_key:
        return {
            "status": "error",
            "error_message": (
                "GOOGLE_MAPS_API_KEY is not configured. Set it in Section 1.1 "
                "and enable the Geocoding API for the project."
            ),
        }

    try:
        response = requests.get(
            GEOCODE_API_URL,
            params={"address": location, "key": api_key},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        payload = response.json()
    except requests.RequestException as exc:
        return {"status": "error", "error_message": f"Geocoding request failed: {exc}"}

    if payload.get("status") != "OK" or not payload.get("results"):
        return {
            "status": "error",
            "error_message": (
                f"Could not geocode '{location}'. Geocoding API status: "
                f"{payload.get('status', 'UNKNOWN')}."
            ),
        }

    top_result = payload["results"][0]
    coordinates = top_result["geometry"]["location"]

    state_code = ""
    for component in top_result.get("address_components", []):
        if "administrative_area_level_1" in component.get("types", []):
            state_code = component.get("short_name", "")
            break

    return {
        "status": "success",
        "location": top_result.get("formatted_address", location),
        "lat": float(coordinates["lat"]),
        "lon": float(coordinates["lng"]),
        "state_code": state_code,
    }

In [10]:
# Washington, DC — a coordinate pair the NWS always covers.
forecast_check = get_weather_forecast(38.8977, -77.0365)
print("get_weather_forecast:", forecast_check["status"])
if forecast_check["status"] == "success":
    print("  location:", forecast_check["location"])
    print("  current: ", forecast_check["current"]["short_forecast"],
          forecast_check["current"]["temperature"] + "F")

alerts_check = get_active_weather_alerts("FL")
print("get_active_weather_alerts:", alerts_check["status"],
      "| active alerts:", alerts_check.get("alert_count"))
for alert in alerts_check.get("alerts", [])[:3]:
    print("  -", alert["event"], "/", alert["severity"], "->", alert["areas"][:60])

geocode_check = get_lat_lon("Denver, CO")
print("get_lat_lon:", geocode_check["status"], "->",
      {k: geocode_check[k] for k in ("lat", "lon", "state_code")}
      if geocode_check["status"] == "success" else geocode_check["error_message"])

get_weather_forecast: success
  location: Washington, DC
  current:  Sunny 85F
get_active_weather_alerts: success | active alerts: 1
  - Heat Advisory / Moderate -> Inland Bay; Calhoun; Inland Gulf; Inland Franklin; Gadsden; 
get_lat_lon: success -> {'lat': 39.7392358, 'lon': -104.990251, 'state_code': 'CO'}


AGENT SETUP

In [11]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a real-time weather alerts assistant for locations in the United States.

TOOLS AND THE ORDER TO USE THEM
1. `get_lat_lon(location)` — ALWAYS call this first to turn the user's place name into
   coordinates. Never guess or recall latitude and longitude from memory.
2. `get_weather_forecast(lat, lon)` — call with the coordinates from step 1 to get current
   conditions and the extended forecast.
3. `get_active_weather_alerts(state_code)` — call with the `state_code` from step 1 to check
   for watches, warnings, and advisories in effect.

If the user names several locations, repeat all three steps for each one.

HOW TO ANSWER
- Lead with any active alert that affects the requested location. Name the event
  (for example, "Heat Advisory"), its severity, and the NWS safety instruction.
- If there is no relevant active alert, say so plainly in one short sentence, then give the
  summary.
- Follow with a two-to-four sentence conditions summary: temperature with units, sky
  conditions, wind, and anything notable in the next day or two.
- Use Fahrenheit, since that is what the NWS returns. Add Celsius only if the user asks.

RULES
- Report only what the tools return. Never invent temperatures, alerts, or forecasts.
- If a tool returns {"status": "error"}, tell the user plainly what failed and what would fix it.
  Do not retry the same failing call more than once.
- The National Weather Service covers only the United States and its territories. For a location
  outside that coverage, say so directly instead of substituting another data source.
- Be concise and factual. No filler and no emoji.
"""

WEATHER_TOOLS = [get_lat_lon, get_weather_forecast, get_active_weather_alerts]

In [12]:
gemini_weather_agent = Agent(
    name="pat_weather_agent_gemini",
    model=MODEL_GEMINI_FLASH,
    description=(
        "Pat, the real-time weather alerts agent. Retrieves live National Weather "
        "Service forecasts and active alerts for US locations."
    ),
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=WEATHER_TOOLS,
)
print("Built:", gemini_weather_agent.name,
      "| tools:", [tool.__name__ for tool in WEATHER_TOOLS])

Built: pat_weather_agent_gemini | tools: ['get_lat_lon', 'get_weather_forecast', 'get_active_weather_alerts']


In [13]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "weather_alerts_app"
USER_ID = "workshop-user-01"

session_service = InMemorySessionService()


async def ask_agent(
    agent: Agent,
    query: str,
    session_id: str,
    verbose: bool = False,
) -> str:
    """
    Send one query to an agent and return its final text response.

    Creates the session if it does not yet exist, streams the run to completion,
    and optionally prints each tool call and tool response for tracing.

    Args:
        agent (Agent): The ADK agent to query.
        query (str): The user's natural-language question.
        session_id (str): Session identifier; reuse it to preserve conversation history.
        verbose (bool): If True, print every tool call and tool result.

    Returns:
        str: The agent's final text response, or an explanatory message if the
            run produced no final text.
    """
    try:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except Exception:
        pass  # Session already exists; continue the existing conversation.

    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=query)])

    final_text = "[no final response produced]"
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if verbose and event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "function_call", None):
                    call = part.function_call
                    print(f"    -> tool call: {call.name}({dict(call.args)})")
                elif getattr(part, "function_response", None):
                    name = part.function_response.name
                    payload = str(part.function_response.response)
                    print(f"    <- tool result: {name} {payload[:110]}...")

        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            if text:
                final_text = text.strip()

    return final_text

In [14]:
answer = await ask_agent(
    gemini_weather_agent,
    "What's the weather like in Denver, Colorado right now?",
    session_id="smoke-test-01",
    verbose=True,
)
print("\n" + "=" * 78)
print(answer)

/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


    -> tool call: get_lat_lon({'location': 'Denver, Colorado'})
    <- tool result: get_lat_lon {'status': 'success', 'location': 'Denver, CO, USA', 'lat': 39.7392358, 'lon': -104.990251, 'state_code': 'CO'...
    -> tool call: get_weather_forecast({'lat': 39.7392358, 'lon': -104.990251})
    -> tool call: get_active_weather_alerts({'state_code': 'CO'})
    <- tool result: get_weather_forecast {'status': 'success', 'location': 'Denver, CO', 'current': {'name': 'Today', 'temperature': '93', 'temperature...
    <- tool result: get_active_weather_alerts {'status': 'success', 'state': 'CO', 'alert_count': 1, 'alerts': [{'event': 'Flash Flood Watch', 'severity': '...

There are no active weather alerts for Denver, Colorado. The current temperature is 93°F and it is mostly sunny with a chance of showers and thunderstorms after 1 PM. Winds are from the southeast at 3 to 7 mph. There is a 40% chance of precipitation, with new rainfall amounts less than a tenth of an inch possible. Showers and 

TESTS

In [15]:
TEST_CITIES = [
    "Seattle, WA",
    "Denver, CO",
    "Miami, FL",
    "Chicago, IL",
    "Phoenix, AZ",
    "New Orleans, LA",
]


async def run_city_tests(agent: Agent, cities: List[str], label: str) -> Dict[str, str]:
    """
    Query an agent about each city in turn and collect the responses.

    Args:
        agent (Agent): The ADK agent under test.
        cities (List[str]): City strings such as "Denver, CO".
        label (str): Short label used in session IDs and printed output.

    Returns:
        Dict[str, str]: Mapping of each city to the agent's response text.
    """
    results: Dict[str, str] = {}
    for index, city in enumerate(cities, start=1):
        query = (
            f"Give me a weather summary for {city}, and tell me about any "
            "active weather alerts there."
        )
        print(f"\n{'=' * 78}\n[{label} {index}/{len(cities)}] {city}\n{'-' * 78}")
        try:
            response = await ask_agent(
                agent, query, session_id=f"{label}-city-{index}"
            )
        except Exception as exc:
            response = f"[RUN FAILED] {type(exc).__name__}: {exc}"
        results[city] = response
        print(textwrap.fill(response, width=96))
    return results


gemini_results = await run_city_tests(gemini_weather_agent, TEST_CITIES, "gemini")


[gemini 1/6] Seattle, WA
------------------------------------------------------------------------------
There are no active weather alerts for Seattle, WA.  The current weather in Seattle, WA is sunny
with a high near 76°F and a north wind of 6 to 10 mph. Tonight will be mostly clear with a low
around 58°F. Sunny conditions are expected tomorrow, with a high near 86°F.

[gemini 2/6] Denver, CO
------------------------------------------------------------------------------
There are no active weather alerts for Denver, CO.  The current temperature in Denver, CO is
93°F, with mostly sunny skies and a chance of showers and thunderstorms after 1 PM. Winds will
be from the east at 3 to 7 mph. Tonight, there's a chance of showers and thunderstorms, with a
low around 62°F. Tuesday will be mostly sunny with a high near 88°F and a chance of showers and
thunderstorms in the afternoon.

[gemini 3/6] Miami, FL
------------------------------------------------------------------------------
There are

In [16]:
def assess_results(results: Dict[str, str], label: str) -> bool:
    """
    Validate that each agent response contains live, tool-sourced weather data.

    Args:
        results (Dict[str, str]): City-to-response mapping from `run_city_tests`.
        label (str): Label for the printed report.

    Returns:
        bool: True if every city passed every check.
    """
    weather_terms = (
        "temperature", "degree", "sunny", "cloud", "rain", "wind", "clear",
        "storm", "humid", "forecast", "high", "low", "snow", "fog", "shower",
    )
    all_passed = True

    print(f"\n{'=' * 78}\nTEST REPORT — {label}\n{'=' * 78}")
    print(f"{'City':<20}{'Non-empty':<12}{'Has temp':<11}{'Weather terms':<16}{'No error':<10}")
    print("-" * 78)

    for city, response in results.items():
        lowered = response.lower()
        non_empty = len(response) > 60
        has_temperature = any(char.isdigit() for char in response)
        has_terms = any(term in lowered for term in weather_terms)
        no_failure = "[run failed]" not in lowered

        passed = non_empty and has_temperature and has_terms and no_failure
        all_passed = all_passed and passed

        def mark(value: bool) -> str:
            return "PASS" if value else "FAIL"

        print(
            f"{city:<20}{mark(non_empty):<12}{mark(has_temperature):<11}"
            f"{mark(has_terms):<16}{mark(no_failure):<10}"
        )

    print("-" * 78)
    print(f"OVERALL: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'} "
          f"({len(results)} cities)")
    return all_passed


gemini_passed = assess_results(gemini_results, "Gemini 2.5 Flash")


TEST REPORT — Gemini 2.5 Flash
City                Non-empty   Has temp   Weather terms   No error  
------------------------------------------------------------------------------
Seattle, WA         PASS        PASS       PASS            PASS      
Denver, CO          PASS        PASS       PASS            PASS      
Miami, FL           PASS        PASS       PASS            PASS      
Chicago, IL         PASS        PASS       PASS            PASS      
Phoenix, AZ         PASS        PASS       PASS            PASS      
New Orleans, LA     PASS        PASS       PASS            PASS      
------------------------------------------------------------------------------
OVERALL: ALL TESTS PASSED (6 cities)


Claude

In [17]:
claude_results: Dict[str, str] = {}

try:
    claude_weather_agent = Agent(
        name="pat_weather_agent_claude",
        model=LiteLlm(
            model=MODEL_CLAUDE, vertex_project=PROJECT_ID, vertex_location=CLAUDE_LOCATION
        ),
        description=(
            "Pat, the real-time weather alerts agent. Retrieves live National Weather "
            "Service forecasts and active alerts for US locations."
        ),
        instruction=WEATHER_AGENT_INSTRUCTIONS,
        tools=WEATHER_TOOLS,
    )
    print("Built:", claude_weather_agent.name)
    # A shorter city list keeps third-party token spend down.
    claude_results = await run_city_tests(claude_weather_agent, TEST_CITIES[:3], "claude")
    claude_passed = assess_results(claude_results, "Claude (Vertex AI Model Garden)")
except Exception as exc:
    print(f"Claude run failed: {type(exc).__name__}: {exc}")
    print(
        "If this looks like an access or model-not-found error, enable Claude for this "
        "project under Vertex AI -> Model Garden -> Claude, and confirm CLAUDE_LOCATION "
        "in Section 1.1 matches a region where it is published."
    )

Built: pat_weather_agent_claude

[claude 1/3] Seattle, WA
------------------------------------------------------------------------------
No active weather alerts affect Seattle directly — the two Air Quality Alerts currently in
effect for Washington are for Chelan, Douglas, Okanogan, and Kittitas counties, none of which
include Seattle/King County.  **Seattle Weather Summary:** - **Today:** Sunny, high near 76°F,
light north wind 6–10 mph. - **Tonight:** Mostly clear, low around 58°F. - **Tuesday:** Warming
up to a sunny 86°F. - **Wednesday–Thursday:** Mostly sunny to partly cloudy, cooling back to the
low-to-mid 70s. - **Friday:** Partly sunny with a chance of light rain moving in after 5 PM,
high near 71°F.  Overall, expect a stretch of dry, sunny weather with a warm spike Tuesday,
before a slight cooldown and possible rain by Friday evening.

[claude 2/3] Denver, CO
------------------------------------------------------------------------------
**No active alerts affecting Denver.** 

SIDE BY SIDE

In [18]:
if claude_results:
    for city in TEST_CITIES[:3]:
        print("=" * 78)
        print(f"CITY: {city}")
        print("-" * 78)
        print("GEMINI 2.5 FLASH:")
        print(textwrap.fill(gemini_results.get(city, "(not run)"), width=96))

        print("\nCLAUDE (VERTEX AI MODEL GARDEN):")
        print(textwrap.fill(claude_results.get(city, "(not run)"), width=96))
        print()
else:
    print("No Claude results to compare. Run Section 5.2 first.")

CITY: Seattle, WA
------------------------------------------------------------------------------
GEMINI 2.5 FLASH:
There are no active weather alerts for Seattle, WA.  The current weather in Seattle, WA is sunny
with a high near 76°F and a north wind of 6 to 10 mph. Tonight will be mostly clear with a low
around 58°F. Sunny conditions are expected tomorrow, with a high near 86°F.

CLAUDE (VERTEX AI MODEL GARDEN):
No active weather alerts affect Seattle directly — the two Air Quality Alerts currently in
effect for Washington are for Chelan, Douglas, Okanogan, and Kittitas counties, none of which
include Seattle/King County.  **Seattle Weather Summary:** - **Today:** Sunny, high near 76°F,
light north wind 6–10 mph. - **Tonight:** Mostly clear, low around 58°F. - **Tuesday:** Warming
up to a sunny 86°F. - **Wednesday–Thursday:** Mostly sunny to partly cloudy, cooling back to the
low-to-mid 70s. - **Friday:** Partly sunny with a chance of light rain moving in after 5 PM,
high near 71°F.  

EDGE CASE

In [19]:
EDGE_CASE_QUERIES = [
    # Outside NWS coverage — the agent should say so rather than fabricate.
    "What's the weather in Paris, France?",
    # Several locations in one turn.
    "Compare the current weather in Boston, MA and San Diego, CA.",
    # Alert-focused phrasing.
    "Are there any severe weather alerts I should know about in Oklahoma City?",
]

for index, query in enumerate(EDGE_CASE_QUERIES, start=1):
    print(f"\n{'=' * 78}\nEDGE CASE {index}: {query}\n{'-' * 78}")
    print(textwrap.fill(
        await ask_agent(gemini_weather_agent, query, session_id=f"edge-{index}"),
        width=96,
    ))


EDGE CASE 1: What's the weather in Paris, France?
------------------------------------------------------------------------------
The National Weather Service covers only the United States and its territories. I cannot provide
weather information for Paris, France.

EDGE CASE 2: Compare the current weather in Boston, MA and San Diego, CA.
------------------------------------------------------------------------------
There are no active weather alerts for Boston, MA. In Boston, MA, it is sunny with a high near
81°F and a southwest wind of 9 to 13 mph. Tonight will be mostly clear with a low around 63°F.
Tuesday will be sunny with a high near 81°F.  There is a Heat Advisory for San Diego County
Coastal Areas in San Diego, CA. Drink plenty of fluids, stay in an air-conditioned room, stay
out of the sun, and check up on relatives and neighbors. In San Diego, CA, there is patchy fog
before 11 am, then sunny, with a high near 85°F and a southwest wind of 0 to 5 mph. Tonight will
have patchy 

In [20]:
# Multi-turn: the follow-up has no city in it, so a correct answer proves
# the session is carrying conversation state.
MEMORY_SESSION = "multi-turn-01"

print("TURN 1")
print(textwrap.fill(await ask_agent(
    gemini_weather_agent, "What's the forecast for Nashville, Tennessee?", MEMORY_SESSION
), width=96))

print("\nTURN 2 (no city named — tests conversational memory)")
print(textwrap.fill(await ask_agent(
    gemini_weather_agent, "Will I need an umbrella there tomorrow?", MEMORY_SESSION
), width=96))

TURN 1
There are no active weather alerts for Nashville, Tennessee.  Today will be sunny with a high
near 90 degrees Fahrenheit, and a north wind of 0 to 5 mph. Tonight will be mostly clear with a
low around 65 degrees Fahrenheit. Tuesday will be sunny with a high near 91 degrees Fahrenheit.

TURN 2 (no city named — tests conversational memory)
No, the forecast for tomorrow (Tuesday) in Nashville, Tennessee is sunny with no mention of
rain.
